# Automated Public Solutions Ensemble



## Why this notebook?

The new notebook voting system has been encouraging blind blending, including blends of blends. It does make it harder to quickly find genuinely good models. Although this notebook has the same flow as all public ensembles (no OOF ensembling and robust CV), it tries to improve the ensembling workflow by:

- Filtering out blind blends and blends of blends using an LLM
- Offering several ensembling strategies
- Fully automated filtering and ensembling - can be run daily via a Kaggle notebook scheduler or locally with a cron job

I hope it demonstrates that blending of public solutions can easily be automated locally and diminishes the voting incentive for manual blind blends as anyone can easily reproduce and automate this process.

## Setup 

(Add-ons > Secrets in your Kaggle notebook):
  1. Add KAGGLE_USERNAME (your Kaggle username)
  2. Add KAGGLE_KEY (your Kaggle API key from kaggle.json)
  3. Add OPENROUTER_API_KEY (optional, only if LLM_FILTER=True)
  4. Set COMPETITION and other config below

# Smart Ensemble - Configuration

## Ensemble Strategy

**Improvements over original:**
- **Dynamic `TOP_N`** — automatically scales with competition age (week 1 → 20, week 2 → 30, week 3 → 40, week 4+ → 50), so you always pull a pool proportional to how many quality notebooks actually exist.
- **Score-weighted reference** — optimization targets now use LB scores extracted from kernel titles as weights, fixing the circular "optimize against the mean you're computing" problem.
- **`ensemble_score_weighted`** — a new dedicated method that directly weights each submission by its extracted LB score.
- **`raise_on_error=False`** in LLM filter — a single unparseable LLM response no longer aborts the whole run.
- **Reliable LLM model** — defaults to `google/gemini-2.5-flash`, which consistently outputs valid JSON.
- **Tighter `MIN_CORR = 0.990`** — removes near-duplicate submissions while keeping genuinely diverse ones.
- **`OPTUNA_TRIALS = 300`** — better convergence for the rank-weighted optimizer.
- **`MAX_SUBS = 10`** — slightly wider selection for more ensemble diversity.
- **Multi-method comparison** — runs `rankweight_optuna`, `logodds`, and `score_weighted` in parallel and reports all three, so you can pick the best before submitting.


In [1]:
from datetime import date
import calendar

# ── Competition ──────────────────────────────────────────────────────────────
COMPETITION = "playground-series-s6e4"   # competition slug
BEST        = "high"                     # "high" = higher score is better (AUC, accuracy, …)

# ── Dynamic TOP_N ─────────────────────────────────────────────────────────────
# Monthly competitions run from the 1st to the last day of the month.
# The pool of quality notebooks grows over time, so we scale TOP_N accordingly.
_today           = date.today()
_start           = date(_today.year, _today.month, 1)
_days_elapsed    = (_today - _start).days + 1          # 1-indexed
_days_in_month   = calendar.monthrange(_today.year, _today.month)[1]
_pct             = _days_elapsed / _days_in_month      # 0..1 progress through month

if _pct <= 0.25:
    TOP_N = 20        # Week 1: few quality notebooks yet
elif _pct <= 0.50:
    TOP_N = 30        # Week 2: pool is growing
elif _pct <= 0.75:
    TOP_N = 40        # Week 3: mature pool
else:
    TOP_N = 50        # Week 4+: full late-competition pool

print(f"Competition day {_days_elapsed}/{_days_in_month} ({_pct*100:.0f}% through month)  →  TOP_N = {TOP_N}")

# ── Selection & ensemble ─────────────────────────────────────────────────────
MAX_SUBS        = 10                     # max submissions to select for ensemble
STRATEGY        = "greedy"               # "greedy" | "cluster" | "top"
MIN_CORR        = 0.990                  # max allowed pairwise correlation (tighter than 0.995)
ENSEMBLE_METHOD = "rankweight_optuna"    # primary method; alternatives compared in step 5
OPTUNA_TRIALS   = 300                    # more trials → better weight optimisation

# ── LLM blend filter ─────────────────────────────────────────────────────────
EXCLUDE_BLENDS = True                    # also exclude by name keywords
LLM_FILTER     = True
LLM_MODEL      = "google/gemini-2.5-flash"   # reliable JSON output; cheap
# LLM_MODEL    = "bytedance-seed/seed-2.0-mini"  # 0.10 $/M — also good
# LLM_MODEL    = "anthropic/claude-sonnet-4.6"   # 5 $/M — highest quality

# ── Display ───────────────────────────────────────────────────────────────────
SHOW_CORR = True
REPORT    = True


Competition day 6/30 (20% through month)  →  TOP_N = 20


In [2]:
import json
import os
import re
import shutil
import subprocess
import tempfile
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd

# Install optuna if needed (for rankweight_optuna)
try:
    import optuna
except ImportError:
    import subprocess as _sp
    _sp.check_call(["pip", "install", "-q", "optuna"])
    import optuna

from scipy.cluster.hierarchy import fcluster, linkage
from scipy.spatial.distance import squareform

# Load secrets from Kaggle Secrets and configure authentication
try:
    from kaggle_secrets import UserSecretsClient
    _secrets = UserSecretsClient()

    # Kaggle CLI authentication (required for kernels list/output/pull)
    os.environ["KAGGLE_USERNAME"] = _secrets.get_secret("KAGGLE_USERNAME")
    os.environ["KAGGLE_KEY"] = _secrets.get_secret("KAGGLE_KEY")
    print("Loaded Kaggle CLI credentials from Kaggle Secrets")

    # OpenRouter API key (only needed for LLM filter)
    if LLM_FILTER:
        try:
            os.environ["OPENROUTER_API_KEY"] = _secrets.get_secret("OPENROUTER_API_KEY")
            print("Loaded OPENROUTER_API_KEY from Kaggle Secrets")
        except Exception as e:
            print(f"Warning: Could not load OPENROUTER_API_KEY: {e}")
            print("LLM filter will be disabled")
            LLM_FILTER = False

except Exception as e:
    raise RuntimeError(
        f"Could not load Kaggle Secrets: {e}\n"
        "Add these secrets in your Kaggle notebook (Add-ons > Secrets):\n"
        "  - KAGGLE_USERNAME: your Kaggle username\n"
        "  - KAGGLE_KEY: your Kaggle API key (from kaggle.json)\n"
        "  - OPENROUTER_API_KEY: (optional, only if LLM_FILTER=True)"
    )

import requests

WORK_DIR = Path("/kaggle/working")
SUBS_DIR = WORK_DIR / "subs_tmp"
OUTPUT_PATH = WORK_DIR / "submission.csv"

Loaded Kaggle CLI credentials from Kaggle Secrets
Loaded OPENROUTER_API_KEY from Kaggle Secrets


In [3]:

def get_kernels_list(competition: str, page_size: int = 50, best: str = "high") -> list:
    """Get list of kernels using kaggle CLI, sorted by score."""
    sort_by = "scoreDescending" if best == "high" else "scoreAscending"
    cmd = ["kaggle", "kernels", "list", "--competition", competition,
           "--page-size", str(page_size), "--sort-by", sort_by, "-v"]
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode != 0:
        print(f"Error listing kernels: {result.stderr}")
        return []

    lines = result.stdout.strip().split('\n')
    if len(lines) < 2:
        return []

    headers = lines[0].split(',')
    kernels = []
    for line in lines[1:]:
        values = line.split(',')
        if len(values) >= len(headers):
            kernel = dict(zip(headers, values))
            kernels.append(kernel)
    return kernels


def get_notebook_source(kernel_ref: str) -> str:
    """Download and read notebook source code using kaggle CLI."""
    with tempfile.TemporaryDirectory() as tmpdir:
        try:
            cmd = ["kaggle", "kernels", "pull", kernel_ref, "-p", tmpdir]
            result = subprocess.run(cmd, capture_output=True, text=True)
            if result.returncode != 0:
                return ""

            for f in Path(tmpdir).iterdir():
                if f.suffix == '.ipynb':
                    with open(f, 'r', encoding='utf-8') as file:
                        notebook = json.load(file)
                    content = []
                    for cell in notebook.get('cells', []):
                        source = ''.join(cell.get('source', []))
                        content.append(source)
                    return '\n\n'.join(content)
                elif f.suffix == '.py':
                    return f.read_text(encoding='utf-8')
            return ""
        except Exception:
            return ""


def download_kernel_output(kernel_ref: str, output_dir: Path) -> Path | None:
    """Download kernel output (submission CSV) using kaggle CLI."""
    try:
        cmd = ["kaggle", "kernels", "output", kernel_ref, "-p", str(output_dir)]
        result = subprocess.run(cmd, capture_output=True, text=True)
        if result.returncode != 0:
            return None
        # Prefer submission.csv if it exists, otherwise first .csv
        csv_files = [f for f in output_dir.iterdir() if f.suffix == '.csv']
        for f in csv_files:
            if f.name == 'submission.csv':
                return f
        return csv_files[0] if csv_files else None
    except Exception:
        return None


def get_sample_submission(competition: str) -> pd.DataFrame | None:
    """Download sample_submission.csv from competition to get expected format."""
    with tempfile.TemporaryDirectory() as tmpdir:
        # Try common sample submission filenames
        for filename in ["sample_submission.csv", "sampleSubmission.csv"]:
            cmd = ["kaggle", "competitions", "download", "-c", competition,
                   "-f", filename, "-p", tmpdir]
            result = subprocess.run(cmd, capture_output=True, text=True)
            if result.returncode == 0:
                fpath = Path(tmpdir) / filename
                if fpath.exists():
                    return pd.read_csv(fpath)
                # Might be zipped
                for f in Path(tmpdir).iterdir():
                    if f.suffix == '.csv':
                        return pd.read_csv(f)
                    elif f.suffix == '.zip':
                        import zipfile
                        with zipfile.ZipFile(f) as zf:
                            for name in zf.namelist():
                                if name.endswith('.csv'):
                                    with zf.open(name) as zfile:
                                        return pd.read_csv(zfile)
    return None


def extract_score_from_title(title: str) -> str:
    """Extract score from kernel title."""
    patterns = [
        r'([0-9]\.[0-9]{3,6})',
        r'([0-9])[_-]([0-9]{3,6})',
    ]
    for pattern in patterns:
        match = re.search(pattern, title.lower())
        if match:
            if len(match.groups()) == 2:
                return f"{match.group(1)}.{match.group(2)}"
            return match.group(1)
    return "-"

In [4]:

def test_llm_connection(api_key: str, model: str) -> tuple[bool, str]:
    """Test LLM connection with a simple request."""
    headers = {
        "Authorization": f"Bearer {api_key}",
        "Content-Type": "application/json",
    }
    data = {
        "model": model,
        "messages": [{"role": "user", "content": "Say 'ok'"}],
        "max_tokens": 10,
    }
    try:
        response = requests.post(
            "https://openrouter.ai/api/v1/chat/completions",
            headers=headers, json=data, timeout=30
        )
        if response.status_code == 401:
            return False, "Invalid API key (401 Unauthorized)"
        elif response.status_code == 402:
            return False, "Payment required - check OpenRouter credits (402)"
        elif response.status_code == 429:
            return False, "Rate limited (429)"
        elif response.status_code != 200:
            return False, f"HTTP {response.status_code}: {response.text[:100]}"
        response.raise_for_status()
        return True, ""
    except requests.exceptions.Timeout:
        return False, "Connection timeout"
    except requests.exceptions.ConnectionError:
        return False, "Connection error"
    except Exception as e:
        return False, f"Unexpected error: {str(e)[:100]}"


def parse_llm_json_response(text: str) -> dict:
    """Robustly parse JSON from LLM response."""
    text = text.strip()

    try:
        return json.loads(text)
    except json.JSONDecodeError:
        pass

    if '```' in text:
        match = re.search(r'```(?:json)?\s*([\s\S]*?)```', text)
        if match:
            try:
                return json.loads(match.group(1).strip())
            except json.JSONDecodeError:
                pass

    match = re.search(r'\{[^{}]*"is_blind_blend"[^{}]*\}', text, re.DOTALL)
    if match:
        try:
            return json.loads(match.group(0))
        except json.JSONDecodeError:
            pass

    is_blend = None
    if re.search(r'"is_blind_blend"\s*:\s*true', text, re.IGNORECASE):
        is_blend = True
    elif re.search(r'"is_blind_blend"\s*:\s*false', text, re.IGNORECASE):
        is_blend = False
    elif re.search(r'\bblind\s*blend\b', text, re.IGNORECASE) and re.search(r'\b(yes|true|is a)\b', text, re.IGNORECASE):
        is_blend = True
    elif re.search(r'\b(not|no|original)\b.*\bblind\s*blend\b', text, re.IGNORECASE):
        is_blend = False

    if is_blend is not None:
        conf_match = re.search(r'"confidence"\s*:\s*"(high|medium|low)"', text, re.IGNORECASE)
        confidence = conf_match.group(1).lower() if conf_match else "medium"
        reason_match = re.search(r'"reason"\s*:\s*"([^"]*)"', text)
        reason = reason_match.group(1)[:100] if reason_match else "parsed from unstructured response"
        return {"is_blind_blend": is_blend, "confidence": confidence, "reason": reason}

    raise json.JSONDecodeError(f"Could not extract JSON from response: {text[:100]}...", text, 0)


def analyze_for_blind_blend(notebook_content: str, api_key: str, model: str, raise_on_error: bool = False) -> dict:
    """Use LLM to detect if notebook is a blind blend."""
    max_chars = 15000
    if len(notebook_content) > max_chars:
        notebook_content = notebook_content[:max_chars] + "\n...[truncated]..."

    prompt = f"""Analyze this Kaggle notebook and determine if it's a blind blend or ensemble of public submissions.

A BLIND BLEND notebook typically:
- Downloads or loads OTHER people's submission CSV files
- Averages, blends, or ensembles predictions from multiple public kernels
- Does NOT train its own model from scratch
- Uses terms like blend, ensemble submissions, and average predictions
- References other kernel names or URLs

An ORIGINAL notebook:
- Trains its own model(s) on the competition data
- May use an ensemble of ITS OWN models (that's fine)
- Does feature engineering and model training

example good orginal notebook: PlaygroundS6E3|Public|Baseline|V1

NOTEBOOK CONTENT:
{notebook_content}

Respond with ONLY a JSON object (no markdown):
{{"is_blind_blend": true/false, "confidence": "high/medium/low", "reason": "brief explanation"}}"""

    headers = {
        "Authorization": f"Bearer {api_key}",
        "Content-Type": "application/json",
    }
    data = {
        "model": model,
        "messages": [{"role": "user", "content": prompt}],
        "max_tokens": 200,
        "temperature": 0.1
    }

    try:
        response = requests.post(
            "https://openrouter.ai/api/v1/chat/completions",
            headers=headers, json=data, timeout=60
        )
        response.raise_for_status()
        result_text = response.json()['choices'][0]['message']['content']
        return parse_llm_json_response(result_text)
    except requests.exceptions.Timeout:
        error_msg = "LLM request timeout"
        if raise_on_error:
            raise RuntimeError(error_msg)
        return {"is_blind_blend": False, "confidence": "low", "reason": error_msg, "error": True}
    except requests.exceptions.HTTPError as e:
        error_msg = f"LLM API error: {e.response.status_code}"
        if raise_on_error:
            raise RuntimeError(f"{error_msg} - {e.response.text[:200]}")
        return {"is_blind_blend": False, "confidence": "low", "reason": error_msg, "error": True}
    except json.JSONDecodeError as e:
        error_msg = f"Failed to parse LLM response: {str(e)[:50]}"
        if raise_on_error:
            raise RuntimeError(error_msg)
        return {"is_blind_blend": False, "confidence": "low", "reason": error_msg, "error": True}
    except Exception as e:
        error_msg = f"LLM error: {str(e)[:80]}"
        if raise_on_error:
            raise RuntimeError(error_msg)
        return {"is_blind_blend": False, "confidence": "low", "reason": error_msg, "error": True}

In [5]:

def download_and_filter_submissions(
    competition: str,
    top_n: int,
    best: str,
    output_dir: Path,
    n_expected_cols: int = 2,
    reference_cols: list = None,
    reference_rows: int = None,
    llm_filter: bool = False,
    llm_model: str = 'google/gemini-2.5-flash'
):
    api_key = os.getenv('OPENROUTER_API_KEY') if llm_filter else None
    if llm_filter and not api_key:
        print('Warning: OPENROUTER_API_KEY not set. Disabling LLM filter.')
        llm_filter = False

    if llm_filter:
        print(f'\nLLM Filter -- model: {llm_model}')
        print('  Testing connection...', end=' ', flush=True)
        success, error = test_llm_connection(api_key, llm_model)
        if not success:
            print(f'FAILED: {error}. Continuing without LLM filter.')
            llm_filter = False
        else:
            print('OK')

    print(f'\nFetching kernels for {competition} (best={best})...')
    kernels = get_kernels_list(competition, page_size=top_n * 3, best=best)
    print(f'Found {len(kernels)} kernels')
    if not kernels:
        print('No kernels found. Check competition slug.')
        return [], []

    output_dir.mkdir(parents=True, exist_ok=True)
    submissions, files, notebook_results = [], [], []
    downloaded = 0

    print(f"\n{'='*80}")
    print(f"{'#':<3} {'Notebook':<40} {'Score':<10} {'Status'}")
    print('-' * 80)

    for kernel in kernels:
        if downloaded >= top_n:
            break

        kernel_ref   = kernel.get('ref', '')
        kernel_title = kernel.get('title', kernel_ref)
        kernel_name  = kernel_ref.split('/')[-1] if '/' in kernel_ref else kernel_ref
        score        = extract_score_from_title(kernel_title)
        result       = {'name': kernel_name, 'score': score, 'status': ''}

        if llm_filter:
            source = get_notebook_source(kernel_ref)
            if source:
                try:
                    analysis = analyze_for_blind_blend(source, api_key, llm_model, raise_on_error=False)
                    if analysis.get('is_blind_blend', False):
                        conf   = analysis.get('confidence', '?')
                        reason = analysis.get('reason', '')[:30]
                        result['status'] = f'BLIND BLEND ({conf})'
                        notebook_results.append(result)
                        print(f"   {kernel_name[:40]:<40} {score:<10} BLIND BLEND ({conf}: {reason})")
                        continue
                except RuntimeError as e:
                    print(f'\n   WARNING: LLM analysis failed for {kernel_name}: {e}. Keeping.')

        kernel_output_dir = output_dir / f'sub_{downloaded+1:02d}_{kernel_name[:30]}'
        kernel_output_dir.mkdir(parents=True, exist_ok=True)
        csv_path = download_kernel_output(kernel_ref, kernel_output_dir)

        if csv_path:
            try:
                df = pd.read_csv(csv_path)

                if len(df.columns) != n_expected_cols:
                    msg = f'SKIP (got {len(df.columns)} cols, expected {n_expected_cols})'
                    result['status'] = msg
                    notebook_results.append(result)
                    print(f'   {kernel_name[:40]:<40} {score:<10} {msg}')
                    shutil.rmtree(kernel_output_dir, ignore_errors=True)
                    continue

                if reference_cols is None:
                    reference_cols = list(df.columns)
                    reference_rows = len(df)

                if len(df) != reference_rows:
                    result['status'] = 'SKIP (rows)'
                    notebook_results.append(result)
                    print(f'   {kernel_name[:40]:<40} {score:<10} SKIP (rows)')
                    shutil.rmtree(kernel_output_dir, ignore_errors=True)
                    continue

                df.columns = reference_cols
                new_path = output_dir / f'sub_{downloaded+1:02d}_{kernel_name[:30]}.csv'
                df.to_csv(new_path, index=False)
                submissions.append(df)
                files.append(new_path)
                downloaded += 1
                result['status'] = 'OK'
                notebook_results.append(result)
                print(f'{downloaded:2d}. {kernel_name[:40]:<40} {score:<10} OK')
                shutil.rmtree(kernel_output_dir, ignore_errors=True)
            except Exception as exc:
                result['status'] = 'FAILED'
                notebook_results.append(result)
                print(f'   {kernel_name[:40]:<40} {score:<10} FAILED ({exc})')
                shutil.rmtree(kernel_output_dir, ignore_errors=True)
        else:
            result['status'] = 'NO OUTPUT'
            notebook_results.append(result)
            print(f'   {kernel_name[:40]:<40} {score:<10} NO OUTPUT')
            shutil.rmtree(kernel_output_dir, ignore_errors=True)

    blind_count = sum(1 for r in notebook_results if 'BLIND BLEND' in r['status'])
    print('-' * 80)
    print(f'Downloaded: {len(submissions)} | Blind blends filtered: {blind_count} | Checked: {len(notebook_results)}')
    print(f"{'='*80}\n")
    return submissions, files


In [6]:

def get_target_matrix(submissions, target_col):
    return np.column_stack([df[target_col].values for df in submissions])


def get_target_matrix_multiclass(submissions, target_cols):
    per_class = {col: np.column_stack([df[col].values.astype(float) for df in submissions])
                 for col in target_cols}
    flat = np.vstack(list(per_class.values()))  # (n_samples * K, n_subs)
    return flat, per_class


def compute_correlation_matrix(targets):
    return np.corrcoef(targets.T)


def is_blend_submission(filename):
    blend_keywords = [
        'blend', 'ensemble', 'stack', 'mix', 'combine', 'merge', 'avg',
        'mean', 'median', 'weighted', 'meta', 'final'
    ]
    return any(kw in filename.lower() for kw in blend_keywords)


def filter_base_models(submissions, files, exclude_blends=True):
    if not exclude_blends:
        return submissions, files, []
    filtered_subs, filtered_files, excluded = [], [], []
    for sub, f in zip(submissions, files):
        if is_blend_submission(f.name):
            excluded.append(f.name)
        else:
            filtered_subs.append(sub)
            filtered_files.append(f)
    return filtered_subs, filtered_files, excluded


def filter_incompatible_scales(submissions, files, target_col, threshold=0.5):
    if target_col is None:  # multiclass_prob: skip this filter
        return submissions, files, []

    sample = submissions[0][target_col]
    if pd.api.types.is_numeric_dtype(sample):
        means = np.array([df[target_col].mean() for df in submissions])
        stds  = np.array([df[target_col].std()  for df in submissions])
        median_mean, median_std = np.median(means), np.median(stds)
        filtered_subs, filtered_files, excluded = [], [], []
        for sub, f, m, s in zip(submissions, files, means, stds):
            if (abs(m - median_mean) / (abs(median_mean) + 1e-10) <= threshold and
                    abs(s - median_std) / (abs(median_std) + 1e-10) <= threshold):
                filtered_subs.append(sub)
                filtered_files.append(f)
            else:
                excluded.append(f.name)
        return filtered_subs, filtered_files, excluded
    else:
        all_cats   = sorted(sample.unique())
        freqs_list = np.array([
            [df[target_col].value_counts().get(c, 0) / len(df) for c in all_cats]
            for df in submissions
        ])
        median_freq = np.median(freqs_list, axis=0)
        filtered_subs, filtered_files, excluded = [], [], []
        for sub, f, freqs in zip(submissions, files, freqs_list):
            if float(np.max(np.abs(freqs - median_freq))) <= threshold:
                filtered_subs.append(sub)
                filtered_files.append(f)
            else:
                excluded.append(f.name)
        return filtered_subs, filtered_files, excluded


# ---- Ordinal encoding -------------------------------------------------------

_KNOWN_ORDINALS = [
    ({'low', 'medium', 'high'},                       ['Low', 'Medium', 'High']),
    ({'low', 'medium', 'high', 'very high'},           ['Low', 'Medium', 'High', 'Very High']),
    ({'none', 'low', 'medium', 'high'},                ['None', 'Low', 'Medium', 'High']),
    ({'no', 'low', 'medium', 'high'},                  ['No', 'Low', 'Medium', 'High']),
    ({'poor', 'fair', 'good', 'excellent'},            ['Poor', 'Fair', 'Good', 'Excellent']),
    ({'very poor', 'poor', 'fair', 'good', 'excellent'},
     ['Very Poor', 'Poor', 'Fair', 'Good', 'Excellent']),
]


def detect_ordinal_encoding(series):
    if pd.api.types.is_numeric_dtype(series):
        return None
    unique_vals  = set(series.unique())
    unique_lower = {v.lower().strip() for v in unique_vals}
    for pattern_set, ordered_labels in _KNOWN_ORDINALS:
        if unique_lower == pattern_set:
            lower_to_canonical = {lbl.lower().strip(): lbl for lbl in ordered_labels}
            enc = {v: ordered_labels.index(lower_to_canonical[v.lower().strip()])
                   for v in unique_vals}
            return enc, ordered_labels
    sorted_vals = sorted(unique_vals)
    print(f'Warning: unrecognised category set -- assuming alphabetical: {sorted_vals}')
    return {v: i for i, v in enumerate(sorted_vals)}, sorted_vals


def encode_submissions(submissions, target_col, encoding_map):
    encoded = []
    for df in submissions:
        df2 = df.copy()
        df2[target_col] = df2[target_col].map(encoding_map)
        if df2[target_col].isna().any():
            n_na = int(df2[target_col].isna().sum())
            print(f'  Warning: {n_na} unmapped values filled with 0')
            df2[target_col] = df2[target_col].fillna(0)
        df2[target_col] = df2[target_col].astype(float)
        encoded.append(df2)
    return encoded


def decode_predictions(result, decoding_list):
    n_classes = len(decoding_list)
    indices   = np.clip(np.round(result).astype(int), 0, n_classes - 1)
    return np.array([decoding_list[i] for i in indices])


# ---- Auto-detect competition type from sample_submission.csv ---------------

def auto_detect_target_type(sample_sub):
    if sample_sub is None or sample_sub.empty:
        print('WARNING: sample_submission.csv unavailable -- defaulting to binary_prob.')
        return {
            'target_type': 'binary_prob', 'id_col': 'id',
            'target_cols': ['target'], 'n_expected_cols': 2,
            'encoding_map': None, 'decoding_list': None, 'is_categorical': False,
        }

    id_col = sample_sub.columns[0]
    n_cols = len(sample_sub.columns)
    config = {
        'id_col': id_col, 'n_expected_cols': n_cols,
        'encoding_map': None, 'decoding_list': None, 'is_categorical': False,
    }

    # Multiple target columns -> multiclass probability
    if n_cols > 2:
        target_cols = list(sample_sub.columns[1:])
        config.update({'target_type': 'multiclass_prob', 'target_cols': target_cols})
        print(f'Detected: multiclass_prob  ({len(target_cols)} classes: {target_cols})')
        print('  Tip: BEST controls kernel sort order but does not affect per-class ensembling.')
        return config

    # Single target column
    target_col  = sample_sub.columns[1]
    config['target_cols'] = [target_col]
    sample_vals = sample_sub[target_col]

    if pd.api.types.is_numeric_dtype(sample_vals):
        min_v = float(sample_vals.min())
        max_v = float(sample_vals.max())
        if min_v >= 0.0 and max_v <= 1.0:
            config['target_type'] = 'binary_prob'
            print(f"Detected: binary_prob    target='{target_col}'  range=[{min_v:.4f}, {max_v:.4f}]")
        else:
            config['target_type'] = 'regression'
            print(f"Detected: regression     target='{target_col}'  range=[{min_v:.4f}, {max_v:.4f}]")
            print('  Tip: for RMSE/log-loss competitions you may want BEST="low".')
    else:
        # String target: sample_submission typically contains only a single
        # placeholder label (e.g. all rows = 'Low'), so we cannot reliably
        # derive the full class set here.  Mark as categorical and defer the
        # ordinal-order detection until we have real submission data.
        config.update({
            'target_type': 'categorical',   # refined after download
            'target_cols': [target_col],
            'is_categorical': True,
            'encoding_map': None,            # determined from submissions
            'decoding_list': None,
        })
        n_unique = sample_vals.nunique()
        print(f"Detected: categorical  target='{target_col}'")
        print(f"  sample_submission has {n_unique} unique value(s): {list(sample_vals.unique())}")
        print('  Full class set will be determined from downloaded submissions.')

    return config


In [7]:

def greedy_diverse_selection(targets, corr_matrix, files, max_subs=10, min_corr_threshold=0.995):
    n_subs = targets.shape[1]
    if n_subs <= max_subs:
        return list(range(n_subs)), []

    selected = [0]
    remaining = set(range(1, n_subs))
    selection_log = [{"idx": 0, "file": files[0].name, "max_corr": None, "reason": "seed (best LB)"}]

    while len(selected) < max_subs and remaining:
        best_idx = None
        best_max_corr = 1.0
        for idx in remaining:
            max_corr = max(corr_matrix[idx, sel] for sel in selected)
            if max_corr < best_max_corr:
                best_max_corr = max_corr
                best_idx = idx
        if best_idx is None or best_max_corr > min_corr_threshold:
            selection_log.append({
                "idx": None, "file": None, "max_corr": best_max_corr,
                "reason": f"stopped: all remaining corr > {min_corr_threshold}"
            })
            break
        selected.append(best_idx)
        remaining.remove(best_idx)
        selection_log.append({
            "idx": best_idx, "file": files[best_idx].name,
            "max_corr": best_max_corr, "reason": "lowest max-correlation"
        })
    return selected, selection_log


def cluster_based_selection(targets, corr_matrix, files, n_clusters=5):
    dist_matrix = 1 - corr_matrix
    np.fill_diagonal(dist_matrix, 0)
    dist_matrix = np.clip((dist_matrix + dist_matrix.T) / 2, 0, None)

    condensed_dist = squareform(dist_matrix)
    linkage_matrix = linkage(condensed_dist, method='average')
    cluster_labels = fcluster(linkage_matrix, n_clusters, criterion='maxclust')

    selected = []
    cluster_info = []
    for cluster_id in range(1, n_clusters + 1):
        cluster_indices = np.where(cluster_labels == cluster_id)[0]
        if len(cluster_indices) > 0:
            best_in_cluster = cluster_indices[0]
            selected.append(best_in_cluster)
            cluster_info.append({
                "cluster_id": cluster_id,
                "size": len(cluster_indices),
                "selected_idx": best_in_cluster,
                "selected_file": files[best_in_cluster].name,
                "members": [files[i].name for i in cluster_indices]
            })
    return selected, cluster_info, cluster_labels


def top_n_selection(targets, files, n=10):
    return list(range(min(n, targets.shape[1])))

In [8]:
# ── helpers ──────────────────────────────────────────────────────────────────

def ensemble_mean(targets, indices):
    return targets[:, indices].mean(axis=1)

def ensemble_median(targets, indices):
    return np.median(targets[:, indices], axis=1)

def ensemble_trimmed_mean(targets, indices, trim=0.1):
    from scipy import stats
    return stats.trim_mean(targets[:, indices], trim, axis=1)

def ensemble_geometric_mean(targets, indices):
    selected = np.clip(targets[:, indices], 1e-10, 1 - 1e-10)
    return np.exp(np.mean(np.log(selected), axis=1))

def ensemble_log_odds(targets, indices):
    """Average in log-odds space — better than simple mean for probabilities."""
    selected = np.clip(targets[:, indices], 1e-10, 1 - 1e-10)
    log_odds  = np.log(selected / (1 - selected))
    avg_log_odds = np.mean(log_odds, axis=1)
    return 1 / (1 + np.exp(-avg_log_odds))

def ensemble_rank_average(targets, indices):
    from scipy import stats
    selected  = targets[:, indices]
    n_samples = selected.shape[0]
    ranks = np.zeros_like(selected)
    for i in range(selected.shape[1]):
        ranks[:, i] = stats.rankdata(selected[:, i]) / n_samples
    return np.mean(ranks, axis=1)

def ensemble_power_mean(targets, indices, p=2):
    selected = np.clip(targets[:, indices], 1e-10, 1)
    return np.power(np.mean(np.power(selected, p), axis=1), 1/p)


# ── score-weighted ensemble (NEW) ─────────────────────────────────────────────

def get_lb_score_weights(files, indices):
    """
    Extract LB scores from kernel filenames and return a normalized weight
    vector.  Submissions with higher LB scores receive proportionally more
    weight.  Returns None if fewer than 2 scores can be parsed (fall back to
    equal weights in the caller).
    """
    scores = []
    for i in indices:
        s = extract_score_from_title(files[i].name)
        try:
            scores.append(float(s))
        except (ValueError, TypeError):
            scores.append(None)

    valid = [s for s in scores if s is not None]
    if len(valid) < 2:
        return None

    min_s, max_s = min(valid), max(valid)
    median_s = np.median(valid)

    normalized = []
    for s in scores:
        if s is None:
            # Unknown score → assign median weight so it's not penalised
            normalized.append((median_s - min_s) / (max_s - min_s + 1e-10) + 0.1)
        else:
            normalized.append((s - min_s) / (max_s - min_s + 1e-10) + 0.1)

    w = np.array(normalized, dtype=float)
    w /= w.sum()
    return w


def ensemble_score_weighted(targets, indices, files):
    """
    Weighted average where each submission's weight is proportional to its
    extracted LB score.  Falls back to simple mean if no scores are found.
    """
    w = get_lb_score_weights(files, indices)
    selected = targets[:, indices]
    if w is None:
        print("  [score_weighted] No LB scores found — falling back to simple mean.")
        return selected.mean(axis=1)
    print("  [score_weighted] weights:", {files[i].name[:30]: f"{wi:.3f}" for i, wi in zip(indices, w)})
    return (selected * w).sum(axis=1)


# ── rank-weighted ensemble ────────────────────────────────────────────────────

def ensemble_rank_weighted(targets, indices, main_weights=None, position_weights=None, asc_ratio=0.3):
    selected = targets[:, indices]
    n_samples, n_subs = selected.shape

    if main_weights is None:
        main_weights = np.array([0.5] + [0.5 / (n_subs - 1)] * (n_subs - 1))
        main_weights = main_weights[:n_subs]
        main_weights /= main_weights.sum()

    if position_weights is None:
        position_weights = np.zeros(n_subs)
        if n_subs >= 2:
            position_weights[0]  =  0.07
            position_weights[-1] = -0.07
            for i in range(1, n_subs - 1):
                position_weights[i] = 0.07 - (0.14 * i / (n_subs - 1))

    def blend_with_sort(ascending):
        result = np.zeros(n_samples)
        for row_idx in range(n_samples):
            row_values    = selected[row_idx, :]
            sorted_idx    = np.argsort(row_values)
            if not ascending:
                sorted_idx = sorted_idx[::-1]
            weighted_sum  = 0.0
            total_weight  = 0.0
            for position, sub_idx in enumerate(sorted_idx):
                eff_w = max(0.001, main_weights[sub_idx] + position_weights[position])
                weighted_sum += row_values[sub_idx] * eff_w
                total_weight += eff_w
            result[row_idx] = weighted_sum / total_weight
        return result

    return asc_ratio * blend_with_sort(True) + (1 - asc_ratio) * blend_with_sort(False)


# ── score-weighted reference (fixes circular optimisation) ────────────────────

def build_reference(targets, all_indices, files):
    """
    Build a quality-weighted reference signal for optimisation.
    Uses LB scores extracted from filenames as weights so the optimiser
    is chasing a signal that reflects actual leaderboard performance rather
    than the circular mean of the predictions being weighted.
    Falls back to simple mean when no scores are available.
    """
    w = get_lb_score_weights(files, all_indices)
    if w is None:
        return targets[:, all_indices].mean(axis=1)
    return (targets[:, all_indices] * w).sum(axis=1)


# ── hill climbing ─────────────────────────────────────────────────────────────

def hill_climbing_weights(targets, indices, files, n_iterations=200, step_size=0.05):
    """Optimise weights via hill climbing against an LB-score-weighted reference."""
    selected_targets = targets[:, indices]
    n_selected = len(indices)
    weights    = np.ones(n_selected) / n_selected

    # FIX: use LB-score-weighted reference instead of circular mean
    reference = build_reference(targets, indices, files)

    def score(w):
        pred = (selected_targets * w).sum(axis=1)
        return np.corrcoef(pred, reference)[0, 1]

    best_score = score(weights)
    for _ in range(n_iterations):
        for i in range(n_selected):
            for delta in [step_size, -step_size]:
                new_w = weights.copy()
                new_w[i] += delta
                new_w = np.clip(new_w, 0.01, 1.0)
                new_w /= new_w.sum()
                s = score(new_w)
                if s > best_score:
                    weights    = new_w
                    best_score = s

    return (selected_targets * weights).sum(axis=1), weights.tolist()


# ── Optuna rank-weighted ───────────────────────────────────────────────────────

def ensemble_rank_weighted_optuna(targets, indices, files, n_trials=300, n_folds=5, seed=42):
    """
    Optimise rank-weighted ensemble weights via Optuna.
    Validation target is an LB-score-weighted reference (not the circular mean).
    """
    optuna.logging.set_verbosity(optuna.logging.WARNING)

    selected   = targets[:, indices]
    n_samples, n_subs = selected.shape

    # FIX: LB-score-weighted reference — not the circular targets.mean()
    reference = build_reference(targets, indices, files)

    np.random.seed(seed)
    fold_indices = np.random.randint(0, n_folds, size=n_samples)

    def objective(trial):
        raw_weights    = [trial.suggest_float(f"w_{i}", 0.01, 1.0) for i in range(n_subs)]
        main_weights   = np.array(raw_weights) / sum(raw_weights)
        position_weights = np.array([trial.suggest_float(f"pos_{i}", -0.15, 0.15) for i in range(n_subs)])
        asc_ratio      = trial.suggest_float("asc_ratio", 0.0, 1.0)

        cv_scores = []
        for fold in range(n_folds):
            val_mask = fold_indices == fold
            val_pred = ensemble_rank_weighted(
                selected[val_mask, :], list(range(n_subs)),
                main_weights, position_weights, asc_ratio
            )
            val_ref  = reference[val_mask]
            if val_ref.std() < 1e-10 or val_pred.std() < 1e-10:
                cv_scores.append(0.0)
            else:
                cv_scores.append(np.corrcoef(val_pred, val_ref)[0, 1])
        return np.mean(cv_scores)

    study = optuna.create_study(
        direction="maximize",
        sampler=optuna.samplers.TPESampler(seed=seed)
    )
    study.optimize(objective, n_trials=n_trials, show_progress_bar=False)

    bp = study.best_params
    main_weights     = np.array([bp[f"w_{i}"] for i in range(n_subs)])
    main_weights    /= main_weights.sum()
    position_weights = np.array([bp[f"pos_{i}"] for i in range(n_subs)])
    asc_ratio        = bp["asc_ratio"]

    result = ensemble_rank_weighted(selected, list(range(n_subs)), main_weights, position_weights, asc_ratio)
    return result, {
        "main_weights":     main_weights.tolist(),
        "position_weights": position_weights.tolist(),
        "asc_ratio":        asc_ratio,
        "cv_score":         study.best_value,
    }


# ---- Multiclass probability ensemble ---------------------------------------

def ensemble_multiclass_prob(submissions, target_cols, files, selected_indices, method='score_weighted'):
    from scipy import stats as _stats
    w = get_lb_score_weights(files, selected_indices) if method == 'score_weighted' else None
    result = {}
    for col in target_cols:
        mat = np.column_stack([submissions[i][col].values.astype(float)
                               for i in selected_indices])
        if method == 'rank':
            n_samples = mat.shape[0]
            ranks = np.zeros_like(mat)
            for j in range(mat.shape[1]):
                ranks[:, j] = _stats.rankdata(mat[:, j]) / n_samples
            result[col] = ranks.mean(axis=1)
        elif w is not None:
            result[col] = (mat * w).sum(axis=1)
        else:
            result[col] = mat.mean(axis=1)
    # Row-normalise so probabilities sum to 1
    total = sum(result[col] for col in target_cols)
    return {col: result[col] / (total + 1e-12) for col in target_cols}


In [9]:
# Step 0: Fetch sample_submission + auto-detect competition type
print(f'Fetching sample_submission for {COMPETITION}...')
sample_sub  = get_sample_submission(COMPETITION)
comp_config = auto_detect_target_type(sample_sub)

TARGET_TYPE    = comp_config['target_type']       # 'binary_prob'|'regression'|'ordinal'|'nominal'|'multiclass_prob'
id_col         = comp_config['id_col']
target_cols    = comp_config['target_cols']       # list; length > 1 for multiclass_prob
target_col     = target_cols[0] if len(target_cols) == 1 else None
IS_CATEGORICAL = comp_config['is_categorical']
IS_MULTICLASS  = TARGET_TYPE == 'multiclass_prob'
decoding_list  = comp_config.get('decoding_list')
encoding_map   = comp_config.get('encoding_map')

# Use sample_sub column names + row count as the reference for all submissions
reference_cols = list(sample_sub.columns)    if sample_sub is not None else None
reference_rows = len(sample_sub)             if sample_sub is not None else None

# Step 1: Download & filter submissions
print(f'\nDownloading from {COMPETITION}  (TOP_N={TOP_N})')
submissions, files = download_and_filter_submissions(
    competition=COMPETITION,
    top_n=TOP_N,
    best=BEST,
    output_dir=SUBS_DIR,
    n_expected_cols=comp_config['n_expected_cols'],
    reference_cols=reference_cols,
    reference_rows=reference_rows,
    llm_filter=LLM_FILTER,
    llm_model=LLM_MODEL,
)
assert submissions, 'No submissions found!'

# Step 2: Detect & encode categorical targets using the actual submission data.
# sample_submission.csv usually has only a single placeholder label, so we
# collect the full class set from the downloaded submissions instead.
if IS_CATEGORICAL:
    all_vals = set()
    for _df in submissions:
        all_vals.update(_df[target_col].dropna().astype(str).unique())
    print(f'\nClasses found in submissions: {sorted(all_vals)}')

    _tmp = pd.Series(sorted(all_vals))
    _enc = detect_ordinal_encoding(_tmp)
    if _enc is not None:
        encoding_map, decoding_list = _enc
        _unique_lower = {v.lower().strip() for v in all_vals}
        _is_known = any(_unique_lower == p for p, _ in _KNOWN_ORDINALS)
        TARGET_TYPE = 'ordinal' if _is_known else 'nominal'
        print(f'Refined type : {TARGET_TYPE}  order: {" < ".join(decoding_list)}')
        print('Encoding:')
        for lbl, idx in sorted(encoding_map.items(), key=lambda x: x[1]):
            print(f'  {idx} -> {lbl}')
        submissions = encode_submissions(submissions, target_col, encoding_map)
    else:
        print('Warning: could not determine encoding -- leaving as-is.')
        IS_CATEGORICAL = False

# Step 3: Filter incompatible scales (pass None for multiclass to skip)
submissions, files, excluded_scale = filter_incompatible_scales(submissions, files, target_col)

# Step 4: Filter blends by name
excluded_blends = []
if EXCLUDE_BLENDS:
    submissions, files, excluded_blends = filter_base_models(submissions, files, exclude_blends=True)
    if excluded_blends:
        print(f'\nExcluded {len(excluded_blends)} blend submissions by name')

assert len(submissions) >= 2, 'Not enough submissions after filtering!'
print(f'\nUsing {len(submissions)} submissions after filtering')

# Step 5: Correlation analysis & local scores
if IS_MULTICLASS:
    targets, per_class_matrices = get_target_matrix_multiclass(submissions, target_cols)
else:
    targets = get_target_matrix(submissions, target_col)
    per_class_matrices = None

corr_matrix = compute_correlation_matrix(targets)
upper_tri   = corr_matrix[np.triu_indices_from(corr_matrix, k=1)]
print(f'\nCorrelation summary:')
print(f'  Min: {upper_tri.min():.4f}  |  Max: {upper_tri.max():.4f}  |  Mean: {upper_tri.mean():.4f}')

all_indices  = list(range(targets.shape[1]))
reference    = build_reference(targets, all_indices, files)
local_scores = [np.corrcoef(targets[:, i], reference)[0, 1] for i in range(targets.shape[1])]

print(f"\n{'='*80}")
print(f"{'#':<3} {'Submission':<45} {'LB Score':<10} {'Local':<10} {'Diversity'}")
print('-' * 80)
sorted_indices = np.argsort(local_scores)[::-1]
for rank, i in enumerate(sorted_indices):
    lb_score  = extract_score_from_title(files[i].name)
    mean_corr = (corr_matrix[i, :].sum() - 1) / (len(files) - 1)
    diversity = 1 - mean_corr
    print(f'{rank+1:2d}. {files[i].name[:45]:<45} {lb_score:<10} {local_scores[i]:.6f}  {diversity:.4f}')
print('-' * 80)
print('Local = corr with LB-weighted reference  |  Diversity = 1 - mean pairwise corr')
print(f"{'='*80}")

if SHOW_CORR:
    print('\nCorrelation matrix (first 10):')
    n_show = min(10, len(files))
    for i in range(n_show):
        row = ' '.join(f'{corr_matrix[i,j]:.3f}' for j in range(n_show))
        print(f'  {files[i].name[:20]:20s} | {row}')


Fetching sample_submission for playground-series-s6e4...
Detected: categorical  target='Irrigation_Need'
  sample_submission has 1 unique value(s): ['Low']
  Full class set will be determined from downloaded submissions.


LLM Filter -- model: google/gemini-2.5-flash
  Testing connection... OK

Fetching kernels for playground-series-s6e4 (best=high)...
Found 60 kernels

#   Notebook                                 Score      Status
--------------------------------------------------------------------------------
   ps-s6e4-ensemble-of-solutions-4          -          BLIND BLEND (high: The notebook explicitly downlo)
   ps-s6e4-ensemble-of-solutions-3          -          BLIND BLEND (high: The notebook explicitly downlo)
 1. ps6e4-14-model-gbdt-ensemble             -          OK
   ps-s6e4-ensemble-of-solutions-4-2        -          BLIND BLEND (high: The notebook explicitly states)
   playgrounds6e4-tunedblend-v1             -          BLIND BLEND (high: The notebook explicitly states)


In [10]:

cluster_info = None
selection_log = None

if STRATEGY == "greedy":
    selected_indices, selection_log = greedy_diverse_selection(
        targets, corr_matrix, files, MAX_SUBS, MIN_CORR
    )
    print(f"\nGreedy diverse selection (target: {MAX_SUBS}):")
    for entry in selection_log:
        if entry['idx'] is not None:
            corr_str = f"(max_corr={entry['max_corr']:.4f})" if entry['max_corr'] else "(seed)"
            print(f"  [{selection_log.index(entry)+1}] {entry['file']} {corr_str}")
        else:
            print(f"  {entry['reason']}")

elif STRATEGY == "cluster":
    selected_indices, cluster_info, _ = cluster_based_selection(
        targets, corr_matrix, files, MAX_SUBS
    )
    print(f"\nCluster-based selection ({MAX_SUBS} clusters):")
    for c in cluster_info:
        print(f"  Cluster {c['cluster_id']}: {c['selected_file']} (from {c['size']} submissions)")

else:  # top
    selected_indices = top_n_selection(targets, files, MAX_SUBS)
    print(f"\nTop-{MAX_SUBS} selection:")
    for i, idx in enumerate(selected_indices):
        print(f"  [{i+1}] {files[idx].name}")

print(f"\nSelected {len(selected_indices)} submissions")


Greedy diverse selection (target: 10):
  [1] sub_02_0-979-cv-single-cat-pairwise-t.csv (seed)
  stopped: all remaining corr > 0.99

Selected 1 submissions


In [11]:
print(f'\nPrimary ensemble method: {ENSEMBLE_METHOD}')
optuna_params     = None
hill_weights      = None
multiclass_result = None
best_method       = ENSEMBLE_METHOD

if IS_MULTICLASS:
    print(f'  Multiclass target ({len(target_cols)} classes) -- ensembling per class.')
    _methods_mc = {}
    for _m in ('score_weighted', 'mean', 'rank'):
        _res  = ensemble_multiclass_prob(submissions, target_cols, files, selected_indices, method=_m)
        _flat = np.concatenate([_res[c] for c in target_cols])
        _ref  = build_reference(targets, selected_indices, files)
        n     = min(len(_flat), len(_ref))
        _sc   = float(np.corrcoef(_flat[:n], _ref[:n])[0, 1])
        _methods_mc[_m] = (_res, _sc)
        print(f'  {_m:<20}  corr={_sc:.6f}')
    best_mc = max(_methods_mc, key=lambda k: _methods_mc[k][1])
    multiclass_result = _methods_mc[best_mc][0]
    print(f'  -> Best: {best_mc}')
    best_method = best_mc

else:
    if ENSEMBLE_METHOD == 'mean':
        result = ensemble_mean(targets, selected_indices)
    elif ENSEMBLE_METHOD == 'median':
        result = ensemble_median(targets, selected_indices)
    elif ENSEMBLE_METHOD == 'trimmed':
        result = ensemble_trimmed_mean(targets, selected_indices)
    elif ENSEMBLE_METHOD == 'geomean':
        result = ensemble_geometric_mean(targets, selected_indices)
    elif ENSEMBLE_METHOD == 'logodds':
        result = ensemble_log_odds(targets, selected_indices)
    elif ENSEMBLE_METHOD == 'rank':
        result = ensemble_rank_average(targets, selected_indices)
    elif ENSEMBLE_METHOD == 'power':
        result = ensemble_power_mean(targets, selected_indices, p=2)
    elif ENSEMBLE_METHOD == 'score_weighted':
        result = ensemble_score_weighted(targets, selected_indices, files)
    elif ENSEMBLE_METHOD == 'hill':
        result, hill_weights = hill_climbing_weights(targets, selected_indices, files)
        print('  Optimised weights:')
        for idx, w in zip(selected_indices, hill_weights):
            print(f'    {files[idx].name}: {w:.4f}')
    elif ENSEMBLE_METHOD == 'rankweight':
        result = ensemble_rank_weighted(targets, selected_indices)
    elif ENSEMBLE_METHOD == 'rankweight_optuna':
        print(f'  Optimising with {OPTUNA_TRIALS} Optuna trials...')
        result, optuna_params = ensemble_rank_weighted_optuna(
            targets, selected_indices, files, n_trials=OPTUNA_TRIALS
        )
        if optuna_params:
            print(f"  Best CV score : {optuna_params['cv_score']:.6f}")
            print(f"  Asc/Desc ratio: {optuna_params['asc_ratio']:.3f}")
            print('  Optimised main weights:')
            for idx, w in zip(selected_indices, optuna_params['main_weights']):
                print(f'    {files[idx].name}: {w:.4f} ({w*100:.1f}%)')
            hill_weights = optuna_params['main_weights']
    else:
        raise ValueError(f'Unknown ENSEMBLE_METHOD: {ENSEMBLE_METHOD}')

    # Multi-method comparison
    _ref = build_reference(targets, selected_indices, files)

    def _corr(pred):
        if pred.std() < 1e-10 or _ref.std() < 1e-10:
            return float('nan')
        return float(np.corrcoef(pred, _ref)[0, 1])

    print(f"\n{'='*60}")
    print('Multi-method comparison (corr with LB-weighted reference):')
    print('-' * 60)
    _methods = {}

    if ENSEMBLE_METHOD == 'rankweight_optuna' and optuna_params is not None:
        _methods['rankweight_optuna'] = (result, optuna_params['cv_score'])
    else:
        _r, _p = ensemble_rank_weighted_optuna(targets, selected_indices, files, n_trials=OPTUNA_TRIALS)
        _methods['rankweight_optuna'] = (_r, _corr(_r))

    if TARGET_TYPE == 'binary_prob':
        _methods['logodds'] = (ensemble_log_odds(targets, selected_indices),
                               _corr(ensemble_log_odds(targets, selected_indices)))
    else:
        _ra = ensemble_rank_average(targets, selected_indices)
        _methods['rank_average'] = (_ra, _corr(_ra))

    _sw = ensemble_score_weighted(targets, selected_indices, files)
    _methods['score_weighted'] = (_sw, _corr(_sw))

    for name, (pred, sc) in _methods.items():
        marker = ' <- PRIMARY' if name == ENSEMBLE_METHOD else ''
        print(f'  {name:<22}  corr={sc:.6f}{marker}')

    print('-' * 60)
    best_method = max(_methods, key=lambda k: _methods[k][1])
    print(f'  Best: {best_method}  (corr={_methods[best_method][1]:.6f})')
    print(f"{'='*60}")
    result = _methods[best_method][0]
    print(f"\n-> Using '{best_method}' predictions for submission.")



Primary ensemble method: rankweight_optuna
  Optimising with 300 Optuna trials...
  Best CV score : 1.000000
  Asc/Desc ratio: 0.732
  Optimised main weights:
    sub_02_0-979-cv-single-cat-pairwise-t.csv: 1.0000 (100.0%)

Multi-method comparison (corr with LB-weighted reference):
------------------------------------------------------------
  [score_weighted] No LB scores found — falling back to simple mean.
  rankweight_optuna       corr=1.000000 <- PRIMARY
  rank_average            corr=0.983943
  score_weighted          corr=1.000000
------------------------------------------------------------
  Best: rankweight_optuna  (corr=1.000000)

-> Using 'rankweight_optuna' predictions for submission.


In [12]:
output_df = submissions[0][[id_col]].copy()

if IS_MULTICLASS:
    for col in target_cols:
        output_df[col] = multiclass_result[col]
    print('\nMulticlass prediction stats:')
    for col in target_cols:
        arr = multiclass_result[col]
        print(f'  {col}: mean={arr.mean():.4f}  std={arr.std():.4f}  [{arr.min():.4f}, {arr.max():.4f}]')

elif IS_CATEGORICAL:
    final_predictions = decode_predictions(result, decoding_list)
    output_df[target_col] = final_predictions
    from collections import Counter
    dist = Counter(final_predictions)
    print('\nPrediction distribution:')
    for lbl in decoding_list:
        count = dist.get(lbl, 0)
        print(f'  {lbl}: {count} ({count/len(final_predictions)*100:.1f}%)')

else:
    output_df[target_col] = result
    print(f'\nStats: min={result.min():.4f}  max={result.max():.4f}  mean={result.mean():.4f}  std={result.std():.4f}')

output_df.to_csv(OUTPUT_PATH, index=False)
print(f'\nSaved: {OUTPUT_PATH}')

if SUBS_DIR.exists():
    shutil.rmtree(SUBS_DIR)
    print(f'Cleaned up: {SUBS_DIR}')

remaining_files = list(WORK_DIR.glob('*'))
print(f'\nFiles in {WORK_DIR}:')
for f in remaining_files:
    print(f'  {f.name} ({f.stat().st_size / 1024:.1f} KB)')

import time
_label  = 'multiclass' if IS_MULTICLASS else best_method
message = f'Smart ensemble: {STRATEGY} + {_label} | {TARGET_TYPE} | top{TOP_N} max{MAX_SUBS}'
cmd     = ['kaggle', 'competitions', 'submit', '-c', COMPETITION, '-f', str(OUTPUT_PATH), '-m', message]
print(f'\nSubmitting to {COMPETITION}...')
sub_result = subprocess.run(cmd, capture_output=True, text=True)
if sub_result.returncode != 0:
    print(f'Submit failed: {sub_result.stderr}')
else:
    print(f'Submitted! Message: {message}')
    print('Waiting for score...', flush=True)
    for attempt in range(12):
        time.sleep(5)
        cmd   = ['kaggle', 'competitions', 'submissions', '-c', COMPETITION, '-v']
        check = subprocess.run(cmd, capture_output=True, text=True)
        if check.returncode == 0:
            lines = check.stdout.strip().split('\n')
            if len(lines) >= 2:
                headers = lines[0].split(',')
                values  = lines[1].split(',')
                if len(values) >= len(headers):
                    sub   = dict(zip(headers, values))
                    score = sub.get('publicScore', '')
                    if score and score != 'None':
                        print(f'\nPUBLIC LB SCORE: {score}')
                        break
                    elif sub.get('status', '') == 'error':
                        print(f"\nSubmission error: {sub.get('errorDescription', 'Unknown')}")
                        break
        print(f'  Waiting... ({(attempt+1)*5}s)')
    else:
        print('\nTimeout. Check Kaggle submissions page.')



Prediction distribution:
  Low: 159441 (59.1%)
  Medium: 100261 (37.1%)
  High: 10298 (3.8%)

Saved: /kaggle/working/submission.csv
Cleaned up: /kaggle/working/subs_tmp

Files in /kaggle/working:
  submission.csv (3204.2 KB)
  __notebook__.ipynb (90.1 KB)

Submitting to playground-series-s6e4...
Submitted! Message: Smart ensemble: greedy + rankweight_optuna | ordinal | top20 max10
Waiting for score...
  Waiting... (5s)
  Waiting... (10s)

PUBLIC LB SCORE: 0.97819
